# EDA — IBM AML HI-Small Dataset

**Week 1, Day 1.** Bootstrap for subsequent EDA work. Mounts Drive, downloads dataset, prepares paths.

**Dataset source:** [IBM Transactions for Anti-Money Laundering (AML)](https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml)

In [1]:
# ============================================================
# Bootstrap — run once per Colab session
# ============================================================
# Mounts Drive, installs packages, sets paths, downloads data.
# Idempotent: safe to run every session.

from google.colab import drive
drive.mount('/content/drive')

import os
import pathlib
import zipfile

DATA_ROOT = pathlib.Path('/content/drive/MyDrive/fincrime-sentinel-data')
RAW = DATA_ROOT / 'raw'
PROCESSED = DATA_ROOT / 'processed'
TYPOLOGY = DATA_ROOT / 'typology_guidance'
SANCTIONS = DATA_ROOT / 'sanctions'

for path in [RAW, PROCESSED, TYPOLOGY, SANCTIONS]:
    path.mkdir(parents=True, exist_ok=True)

# Kaggle auth
os.environ['KAGGLE_CONFIG_DIR'] = str(DATA_ROOT)
kaggle_token = DATA_ROOT / 'kaggle.json'
assert kaggle_token.exists(), (
    f"Upload kaggle.json to {DATA_ROOT} before continuing. "
    "Get it from kaggle.com → Settings → Create New API Token."
)
os.chmod(kaggle_token, 0o600)

# Install non-default packages
!pip install -q kaggle pyarrow duckdb networkx

# Download dataset if not already in Drive
trans_file = RAW / 'HI-Small_Trans.csv'
patterns_file = RAW / 'HI-Small_Patterns.txt'

if not trans_file.exists():
    print("Downloading IBM AML dataset zip (~7.6GB, 5-10 min)...")
    !kaggle datasets download \
        -d ealtman2019/ibm-transactions-for-anti-money-laundering-aml \
        -p {RAW}

    zip_path = RAW / 'ibm-transactions-for-anti-money-laundering-aml.zip'

    print("Extracting HI-Small files only...")

    # Selectively extract only the two files
    with zipfile.ZipFile(zip_path, 'r') as z:
        all_files = z.namelist()
        print(f"Files in zip: {all_files}")

        for target in ['HI-Small_Trans.csv', 'HI-Small_Patterns.txt']:
            match = [f for f in all_files if target in f]
            if match:
                print(f"Extracting {match[0]}...")
                z.extract(match[0], path=RAW)
            else:
                print(f"WARNING: {target} not found in zip")

    print("Deleting zip to free space...")
    zip_path.unlink()
    print(f"Deleted {zip_path}")

else:
    print(f"Dataset already in Drive at {RAW}")

# Verify both files exist
assert trans_file.exists(), "HI-Small_Trans.csv missing after download"
patterns_file = RAW / 'HI-Small_Patterns.txt'
assert patterns_file.exists(), "HI-Small_Patterns.txt missing after download"

trans_size_mb = trans_file.stat().st_size / (1024 * 1024)
print(f"\n✓ Bootstrap complete")
print(f"✓ Transactions file: {trans_size_mb:.1f} MB")
print(f"✓ Patterns file: {patterns_file.stat().st_size} bytes")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset already in Drive at /content/drive/MyDrive/fincrime-sentinel-data/raw

✓ Bootstrap complete
✓ Transactions file: 453.6 MB
✓ Patterns file: 323844 bytes


In [2]:
import pathlib
RAW = pathlib.Path('/content/drive/MyDrive/fincrime-sentinel-data/raw')
for f in sorted(RAW.iterdir()):
    size_mb = f.stat().st_size / (1024*1024)
    print(f"{f.name}: {size_mb:.1f} MB")

HI-Small_Patterns.txt: 0.3 MB
HI-Small_Trans.csv: 453.6 MB


In [3]:
import pandas as pd

if not (PROCESSED / 'transactions.parquet').exists():
    print("Converting CSV to Parquet...")

    df = pd.read_csv(
        RAW / 'HI-Small_Trans.csv',
        dtype={
            'From Bank': 'int32',
            'To Bank': 'int32',
            'Account': 'string',
            'Account.1': 'string',
            'Amount Received': 'float64',
            'Receiving Currency': 'category',
            'Amount Paid': 'float64',
            'Payment Currency': 'category',
            'Payment Format': 'category',
            'Is Laundering': 'int8',
        },
        parse_dates=['Timestamp'],
    )

    # Standardise column names to snake_case
    df.columns = (
        df.columns.str.lower()
        .str.replace(' ', '_')
        .str.replace('.', '_', regex=False)
    )

    # Save with snappy compression — fast read/write, good ratio
    df.to_parquet(
        PROCESSED / 'transactions.parquet',
        compression='snappy',
        index=False,
    )

    print(f" Wrote {PROCESSED / 'transactions.parquet'}")
    print(f" Rows: {len(df):,}")
    print(f" Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
else:
    print(f"Parquet already exists at {PROCESSED / 'transactions.parquet'}")

Parquet already exists at /content/drive/MyDrive/fincrime-sentinel-data/processed/transactions.parquet


In [4]:
parquet_path = PROCESSED / 'transactions.parquet'
parquet_size_mb = parquet_path.stat().st_size / (1024**2)
print(f"Parquet file on disk: {parquet_size_mb:.1f} MB")

Parquet file on disk: 127.1 MB


In [5]:
import duckdb

# Load Parquet via DuckDB — much faster than pandas for analytics
con = duckdb.connect()
con.execute(f"CREATE VIEW transactions AS SELECT * FROM '{PROCESSED}/transactions.parquet'")

# Quick sanity check via SQL
con.execute("SELECT COUNT(*) AS rows, COUNT(DISTINCT from_bank) AS banks FROM transactions").df()

,rows,banks
0,5078345,30470


## Schema deep-dive

Every column documented: type, range, cardinality, null rate, meaning.

In [6]:
con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT from_bank) AS unique_from_banks,
        MIN(from_bank) AS min_id,
        MAX(from_bank) AS max_id,
        SUM(CASE WHEN from_bank IS NULL THEN 1 ELSE 0 END) AS null_count
    FROM transactions
""").df()

,total_rows,unique_from_banks,min_id,max_id,null_count
0,5078345,30470,1,356303,0.0


### `from_bank` — sending bank ID

- **Type:** int32
- **Cardinality:** ~30,470 unique ID's
- **Range:** 1 to ~356303
- **Nulls:** 0
- **Meaning:** Identifier for the bank where the sender holds the account.
  Combined with `account_from`, uniquely identifies a sender account globally.

In [7]:
con.execute("""
    SELECT
        -- Timestamp personality
        MIN(timestamp)                  AS ts_min,
        MAX(timestamp)                  AS ts_max,
        COUNT(DISTINCT DATE_TRUNC('day', timestamp)) AS ts_distinct_days,
    FROM transactions
""").df()

,ts_min,ts_max,ts_distinct_days
0,2022-09-01,2022-09-18 16:18:00,18


### 'timestamp' - Date and Datetimes
- **Type:** Datetime
- **Range:** 2022-09-01 to 2022-09-18 16:18:00
- **Distinct days:** 18
- **Nulls:** 0
- **Meaning:** Transaction timestamp at minute precision.
  18-day synthetic window in September 2022.

In [8]:
con.execute("""
    SELECT
        -- Account_from and Account_to personality
        COUNT(DISTINCT account)       AS account_from_unique, -- sender
        COUNT(DISTINCT account_1)  AS account_to_unique, -- receiver
    FROM transactions
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,account_from_unique,account_to_unique
0,496995,420636


### `account` — sender account (origin)
- **Type:** string (hexadecimal format e.g. 1004286A8)
- **Cardinality:** 496,995 unique senders
- **Nulls:** 0
- **Note:** 93.2% of account IDs are hexadecimal (contain A-F).
  Must be stored as string — numeric dtypes corrupt hex values.

### `account_1` — receiver account
- **Type:** string (hexadecimal format)
- **Cardinality:** 420,636 unique receivers
- **Nulls:** 0
- **Note:** 76,359 fewer unique receivers than senders, meaning
  ~76K accounts appear only as senders, never as receivers.

In [9]:
con.execute("""
    SELECT
        -- to_bank personality
        COUNT(DISTINCT to_bank)         AS to_bank_unique,
        MIN(to_bank)                    AS to_bank_min,
        MAX(to_bank)                    AS to_bank_max,
        SUM(CASE WHEN to_bank IS NULL THEN 1 ELSE 0 END) AS null_count
     FROM transactions
""").df()

,to_bank_unique,to_bank_min,to_bank_max,null_count
0,15811,1,356294,0.0


### `to_bank` — receiving institution ID
- **Type:** int32
- **Cardinality:** 15811 unique IDs
- **Range:** 1 to 356294
- **Nulls:** 0
- **Meaning:** Identifier for the bank where the receiver holds the account.

In [10]:
con.execute("""
    SELECT
        -- amount_paid personality
        MIN(amount_paid)                AS paid_min,
        MAX(amount_paid)                AS paid_max,
        PERCENTILE_CONT(0.5) WITHIN GROUP
            (ORDER BY amount_paid)      AS paid_median
     FROM transactions
""").df()

,paid_min,paid_max,paid_median
0,0.000001,1.046302e+12,1414.54


### `amount_paid` — Amount that has been sent
- **Type:** float64
- **Min:** 0.000001
- **Max:** 1.046 trillion
- **Median:** $1,414.54 ← use this as the representative amount
- **Meaning:** Outbound amount in the sender's currency.

In [11]:
con.execute("""
    SELECT
        -- amount_received personality
        MIN(amount_received)            AS received_min,
        MAX(amount_received)            AS received,
        PERCENTILE_CONT(0.5) WITHIN GROUP
            (ORDER BY amount_received)  AS received_median
     FROM transactions
""").df()

,received_min,received,received_median
0,0.000001,1.046302e+12,1411.01


### `amount_received` — Amount that has been received
- **Type:** float64
- **Min:** 0.000001
- **Max:** 1.046 trillion
- **Median:** 1411.01
- **Meaning:** Inbound amount in the receivers currency.

In [12]:
# Receiving currency
con.execute("""
    SELECT receiving_currency, COUNT(*) AS count,
        SUM(is_laundering) * 100.0 / COUNT(*) AS laundering_rate_pct
    FROM transactions
    GROUP BY receiving_currency ORDER BY count DESC
""").df()

,receiving_currency,count,laundering_rate_pct
0,US Dollar,1879341,0.101738
1,Euro,1172017,0.117063
2,Swiss Franc,237884,0.081132
3,Yuan,206551,0.089082
4,Shekel,194988,0.048721
5,Rupee,192065,0.086950
6,UK Pound,181255,0.072826
7,Ruble,157361,0.084519
8,Yen,156319,0.099156
9,Bitcoin,148151,0.037799


### `receiving_currency` — Receiver currency type
- **Type:** category
- **Unique currency:** 13
- **Meaning:** Currency in which the receiving account was credited.

In [13]:
# Payment currency
con.execute("""
    SELECT payment_currency, COUNT(*) AS count,
        SUM(is_laundering) * 100.0 / COUNT(*) AS laundering_rate_pct
    FROM transactions
    GROUP BY payment_currency ORDER BY count DESC
""").df()

,payment_currency,count,laundering_rate_pct
0,US Dollar,1895172,0.100888
1,Euro,1168297,0.117436
2,Swiss Franc,234860,0.082177
3,Yuan,213752,0.086081
4,Shekel,192184,0.049432
5,Rupee,190202,0.087801
6,UK Pound,180738,0.073034
7,Yen,155209,0.099865
8,Ruble,155178,0.085708
9,Bitcoin,146066,0.038339


### `payment_currency` — Payment currency type
- **Type:** category
- **Unique currency:** 13
- **Meaning:** Currency in which the sending account was debited.

In [14]:
# Payment format
con.execute("""
    SELECT payment_format, COUNT(*) AS count,
        SUM(is_laundering) * 100.0 / COUNT(*) AS laundering_rate_pct
    FROM transactions
    GROUP BY payment_format ORDER BY count DESC
""").df()

,payment_format,count,laundering_rate_pct
0,Cheque,1864331,0.017379
1,Credit Card,1323324,0.015567
2,ACH,600797,0.746175
3,Cash,490891,0.022001
4,Reinvestment,481056,0.000000
5,Wire,171855,0.000000
6,Bitcoin,146091,0.038332


### `payment_format` — Mode of payment
- **Type:** category
- **Unique format:** 7 (includes Bitcoin — significant for AUSTRAC
  VASP obligations and crypto-related AML rules)
- **Meaning:** Mode of transaction

In [15]:
con.execute("""
    SELECT
        is_laundering,
        COUNT(*) AS count,
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER () AS pct
    FROM transactions
    GROUP BY is_laundering
""").df()

,is_laundering,count,pct
0,0,5073168,99.898057
1,1,5177,0.101943


### `is_laundering` — laundering or not

- **Type:** int8
- **Cardinality:** 2
- **Laundering false:** 5,073,168 (99.898%)
- **Laundering true:** 5,177 (0.102%)
- **Meaning:** States if the amount is being laundered or not